# Hotel Booking Cancellation — v2 train/test split (group-aware)

v1 split the deduplicated table with a plain random 80/20 split. Deduplication ran before
`arrival_date_year` and `arrival_date_week_number` were dropped, so some rows became identical
afterwards and **93 test rows had an exact twin in training**. A model can score those rows by memory.

v2 keeps the same population and the same columns and changes one thing: the split. Rows with an
identical booking profile are kept together, so a profile is either all in train or all in test.
No new features, no new EDA, nothing is fitted here.

In [1]:
import hashlib, json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

## 1. Same population and columns as v1

In [2]:
df = pd.read_csv('../../data/processed/hotel_bookings_clean.csv')

# identical to notebook v1/02: target out, year and week number dropped, agent is an ID
X = df.drop(columns=['is_canceled', 'arrival_date_year', 'arrival_date_week_number'])
y = df['is_canceled']
X['agent'] = X['agent'].astype(str)
print(X.shape, 'cancel rate:', round(y.mean(), 4))

(84969, 25) cancel rate: 0.2775


## 2. The group key: the booking profile

A profile is the full row of predictors — every column the model sees, and nothing else. The target
is not part of the key, so the grouping cannot peek at the answer. Two rows with the same profile look
exactly the same to a model, so they must land on the same side of the split.

In [3]:
profile = pd.util.hash_pandas_object(X, index=False)
groups = profile.factorize()[0]
sizes = pd.Series(groups).value_counts()
print(f'{len(X):,} rows, {len(sizes):,} distinct profiles, {(sizes > 1).sum():,} profiles shared by 2+ rows '
      f'({sizes[sizes > 1].sum():,} rows)')

84,969 rows, 84,656 distinct profiles, 313 profiles shared by 2+ rows (626 rows)


## 3. How bad was v1?

Recount the overlap in the saved v1 files, using the same profile definition.

In [4]:
v1_train = pd.read_csv('../../data/processed/train.csv', dtype={'agent': str})
v1_test = pd.read_csv('../../data/processed/test.csv', dtype={'agent': str})

def profiles(frame):
    return pd.util.hash_pandas_object(frame[X.columns], index=False)

v1_overlap = profiles(v1_test).isin(set(profiles(v1_train)))
print('v1 test rows whose profile also appears in train:', int(v1_overlap.sum()))

v1 test rows whose profile also appears in train:

 93


## 4. Group-aware split

`StratifiedGroupKFold` with 5 folds; the first fold becomes the test set, which gives the same 80/20
proportion as v1. Stratification keeps the cancellation rate close on both sides, and seed 42 is the
seed used everywhere else in the project. The fold choice is fixed in advance, not picked by looking at
test scores.

In [5]:
SEED = 42
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgkf.split(X, y, groups))

train = X.iloc[train_idx].assign(is_canceled=y.iloc[train_idx])
test = X.iloc[test_idx].assign(is_canceled=y.iloc[test_idx])

for name, part in [('train', train), ('test', test)]:
    print(f'{name}: {len(part):,} rows ({len(part) / len(X):.1%})  cancel rate {part.is_canceled.mean():.4f}')

train: 67,974 rows (80.0%)  cancel rate 0.2775
test: 16,995 rows (20.0%)  cancel rate 0.2775


## 5. Check: no profile crosses the split

In [6]:
shared_groups = set(groups[train_idx]) & set(groups[test_idx])
v2_overlap = int(profiles(test).isin(set(profiles(train))).sum())
print('groups on both sides:', len(shared_groups))
print('v2 test rows whose profile also appears in train:', v2_overlap, '(v1:', int(v1_overlap.sum()), ')')
assert len(shared_groups) == 0 and v2_overlap == 0
assert len(train) + len(test) == len(X)

groups on both sides: 0
v2 test rows whose profile also appears in train: 0 (v1: 93 )


## 6. Save the split and a manifest

The manifest records what is needed to check later that the files were not changed: row counts,
cancellation rates, the seed, how the group key was built, and a SHA-256 hash of each file.

In [7]:
out = Path('../../data/processed/v2')
out.mkdir(parents=True, exist_ok=True)
train.to_csv(out / 'train.csv', index=False)
test.to_csv(out / 'test.csv', index=False)

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

manifest = {
    'source': 'data/processed/hotel_bookings_clean.csv',
    'source_sha256': sha256('../../data/processed/hotel_bookings_clean.csv'),
    'population': 'v1 cleaned table, duplicates removed (notebook 01)',
    'dropped_columns': ['arrival_date_year', 'arrival_date_week_number'],
    'target': 'is_canceled',
    'group_key': 'exact match on all predictor columns (every column except is_canceled), agent as text',
    'splitter': 'StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42), fold 0 = test',
    'seed': SEED,
    'rows': {'train': len(train), 'test': len(test)},
    'cancel_rate': {'train': round(train.is_canceled.mean(), 4), 'test': round(test.is_canceled.mean(), 4)},
    'profile_overlap': {'v1': int(v1_overlap.sum()), 'v2': v2_overlap},
    'files': {f: sha256(out / f) for f in ['train.csv', 'test.csv']},
}
(out / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

{
  "source": "data/processed/hotel_bookings_clean.csv",
  "source_sha256": "c5da72b9c71a368329eb708c4a7fd87ce3389b141a0e3242026ec3182868bc42",
  "population": "v1 cleaned table, duplicates removed (notebook 01)",
  "dropped_columns": [
    "arrival_date_year",
    "arrival_date_week_number"
  ],
  "target": "is_canceled",
  "group_key": "exact match on all predictor columns (every column except is_canceled), agent as text",
  "splitter": "StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42), fold 0 = test",
  "seed": 42,
  "rows": {
    "train": 67974,
    "test": 16995
  },
  "cancel_rate": {
    "train": 0.2775,
    "test": 0.2775
  },
  "profile_overlap": {
    "v1": 93,
    "v2": 0
  },
  "files": {
    "train.csv": "9f71e4c15bb6f0c2d1a4680d12c35a1429ae2db4cfdf400aa8d326e3d33a616c",
    "test.csv": "cc26be9130a77845a5d19b53d26e8549106629bacf9e8884530dbd38f57a82fc"
  }
}


Later v2 notebooks read `data/processed/v2/train.csv` (with `dtype={'agent': str}`) and must use
the same profile groups for cross-validation inside training, so a profile never sits in both the
fitting and the validation folds either. The test file stays closed until the final single pass.